# 00 · Getting the data

Run this notebook top to bottom once. It gets the data onto your machine and shows how to use it.
After that, any notebook can load the data with three lines (section 4).

## 1. Our goal

**Does the text of a company's annual report predict how volatile its share price will be over
the next year, beyond what last year's volatility already tells us?**

Last year's volatility is already a strong predictor, so that is our baseline. We build neural
networks that read the report and see whether they beat it. The full plan is in
[`Oscar/PLAN.md`](../Oscar/PLAN.md).

## 2. What the data is

**27,142 annual reports** from US public companies, filed 1996–2006. For each report we have:

- **Two numbers about the share price:**
  - `logvol_before`: volatility over the 12 months before the report. **The baseline input.**
  - `logvol_after`: volatility over the 12 months after. **What we predict.**

  Both are logs, which is why they're negative (around −3.4 typically).
- **Two sections of the report's text, already turned into numbers:**
  - **Item 1**, the business description: what the company does, its markets and competitors.
  - **Item 7**, management's discussion of the year.

  Each section is cut into **chunks** of about 400 words, and each chunk has been turned into a
  list of **768 numbers** that captures what it says. A typical section is 14–16 chunks, at most 64.
  So each report's text is a short sequence of 768-number vectors per section.

## 3. How to get it

Everything is in **one file, `data.pkl`** (about 1.6 GB).

**Step 1: add the shared OneDrive folder.** Oscar has shared a OneDrive folder called **`sprint2-data`**
with you (check your Bristol email or OneDrive's *Shared* page). Open it in OneDrive on the web and
click **Add shortcut to My files**. It then syncs to your computer like your other OneDrive files.
Right-click `data.pkl` and choose **Always keep on this device** so the copy below doesn't stall.
Don't edit the files in it: it's everyone's copy.

**Step 2: set up Python**, if you haven't already. From the repo root:
`python3 -m venv .venv`, `source .venv/bin/activate`, `pip install -r requirements.txt`.

**Step 3: run the cell below.** It finds the folder and copies `data.pkl` into `data/processed/` in
your copy of the repo, which is where our code looks. If it can't find the folder, paste the folder's
path into `DATA_DIR` (on a Mac: right-click the folder in Finder, hold Option, *Copy as Pathname*).
Never commit the file: `data/processed/` is in `.gitignore` already.

In [ ]:
import shutil
from pathlib import Path

DATA_DIR = None   # e.g. Path("/Users/you/OneDrive - University of Bristol/sprint2-data") if not found

ROOT = Path.cwd().parent   # the repo root is the folder above this notebook's folder

def find_data():
    home = Path.home()
    onedrives = [home / "OneDrive - University of Bristol",
                 home / "Library" / "CloudStorage" / "OneDrive-UniversityofBristol"]
    for od in onedrives:
        for candidate in [od / "sprint2-data", od / "DST" / "sprint2-data"]:
            if (candidate / "data.pkl").exists():
                return candidate
    return None

src = Path(DATA_DIR) if DATA_DIR else find_data()
assert src is not None, "Folder not found: set DATA_DIR above."

dest = ROOT / "data" / "processed" / "data.pkl"
dest.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(src / "data.pkl", dest)
print(f"copied data.pkl  {dest.stat().st_size / 1e9:.2f} GB")

## 4. Loading it

In any notebook, load everything with:

In [ ]:
import pickle
import numpy as np

with open(ROOT / "data" / "processed" / "data.pkl", "rb") as f:
    d = pickle.load(f)

X, y = d["X"], d["y"]
print(f"X {X.shape}, y {y.shape}")
assert X.shape == (27142, 1536), "Something is missing: ask Oscar."

`d` is a dictionary. Every entry has one item per report, all in the same order:

| Key | What it is |
|---|---|
| `d["y"]` | `logvol_after`: volatility over the 12 months after the report. **What we predict.** |
| `d["baseline"]` | `logvol_before`: volatility over the 12 months before. **The number to beat.** |
| `d["year"]` | filing year, for splitting into train and test |
| `d["X"]` | one row of 1536 numbers per report: Item 1's chunks averaged (columns 0–767), then Item 7's (768–1535). Use this with scikit-learn or a simple neural network. |
| `d["X_item1"][i]` | report `i`'s Item 1 as a sequence of chunks, shape (chunks × 768). For the attention network. |
| `d["X_item7"][i]` | the same for Item 7 |
| `d["info"]` | a table of company name, date and SEC number, to see which company a row is |

For example, report 0's Item 7 as a sequence of chunks:

In [ ]:
i = 0
print(d["info"].loc[i, "name"], d["year"][i], "| Item 7 as", d["X_item7"][i].shape, "(chunks x 768)")

**Training and testing.** We always train on earlier years and test on a later one, so a model
never sees the future: for example, train on 2001–2005 and test on 2006. Models are scored by mean
squared error (MSE) on `y`. The number to beat is the baseline that predicts next year's
volatility is the same as last year's:

In [ ]:
year = d["year"]
train = (year >= 2001) & (year <= 2005)
test = year == 2006

mse = np.mean((y[test] - d["baseline"][test]) ** 2)
print(f"train {train.sum():,} reports, test {test.sum():,} reports")
print(f"Baseline 'same as last year' MSE on 2006: {mse:.4f}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(d["baseline"], y, s=2, alpha=0.2)
lims = [-8, 0]
ax.plot(lims, lims, color="k", lw=1)
ax.set(xlim=lims, ylim=lims, xlabel="log volatility, year before",
       ylabel="log volatility, year after", title="Last year's volatility is a strong baseline")
plt.show()